In [8]:
import pandas as pd
from collections import Counter
from pathlib import Path
import json

In [9]:
# ---------- CONFIG ----------
OUTPUT_RESULTS = Path(r"..\data\results")

# ---------- CARGAR LOS ARCHIVOS GENERADOS ----------
path_corpus = OUTPUT_RESULTS / "corpus_cleaned_POS.csv.gz"
path_ciencia = OUTPUT_RESULTS / "ciencia_chunks_POS.csv.gz"


In [10]:
corpus_POS = pd.read_csv(path_corpus, compression="gzip")

for col in ["verbos", "adjetivos", "sustantivos"]:
    corpus_POS[col] = corpus_POS[col].apply(json.loads)

In [11]:
ciencia_POS = pd.read_csv(path_ciencia, compression="gzip")

for col in ["verbos", "adjetivos", "sustantivos"]:
    ciencia_POS[col] = ciencia_POS[col].apply(json.loads)

In [12]:
# ---------- FUNCIÓN DE COMPARACIÓN ----------
def comparar_frecuencias_pos(df_corpus, df_ciencia, columna_pos):
    """
    Compara las frecuencias de una categoría POS (verbos, sustantivos o adjetivos)
    entre corpus general y fragmentos de ciencia.
    
    Devuelve un DataFrame con:
      palabra | freq_corpus | freq_ciencia | relacion
    """
    # Combinar todos los diccionarios de frecuencias en cada conjunto
    total_corpus = Counter()
    for d in df_corpus[columna_pos].dropna():
        total_corpus.update(d)

    total_ciencia = Counter()
    for d in df_ciencia[columna_pos].dropna():
        total_ciencia.update(d)

    # Crear DataFrames
    df_corpus_freq = pd.DataFrame(total_corpus.items(), columns=["palabra", "freq_corpus"])
    df_ciencia_freq = pd.DataFrame(total_ciencia.items(), columns=["palabra", "freq_ciencia"])

    # Unir por palabra
    df_merge = pd.merge(df_corpus_freq, df_ciencia_freq, on="palabra", how="outer").fillna(0)

    # Calcular relación
    df_merge["relacion"] = df_merge["freq_ciencia"] / (df_merge["freq_corpus"] + 1e-6)

    # Ordenar por relación descendente
    df_merge = df_merge.sort_values(by="relacion", ascending=False).reset_index(drop=True)

    return df_merge

In [13]:
# ---------- APLICAR LA FUNCIÓN A LOS TRES TIPOS ----------
tabla_verbos = comparar_frecuencias_pos(corpus_POS, ciencia_POS, "verbos")
tabla_sustantivos = comparar_frecuencias_pos(corpus_POS, ciencia_POS, "sustantivos")
tabla_adjetivos = comparar_frecuencias_pos(corpus_POS, ciencia_POS, "adjetivos")

In [14]:
print("\n=== Verbos más sobrerrepresentados en ciencia ===")
print(tabla_verbos.head(10))

print("\n=== Sustantivos más sobrerrepresentados en ciencia ===")
print(tabla_sustantivos.head(10))

print("\n=== Adjetivos más sobrerrepresentados en ciencia ===")
print(tabla_adjetivos.head(10))


=== Verbos más sobrerrepresentados en ciencia ===
         palabra  freq_corpus  freq_ciencia  relacion
0  descarbonizar            5           5.0       1.0
1      presioner            4           4.0       1.0
2          ensar            4           4.0       1.0
3          noter            3           3.0       1.0
4  manifestémono            3           3.0       1.0
5  mercantilizar            3           3.0       1.0
6     enverdecer            3           3.0       1.0
7       inflegir            3           3.0       1.0
8      amartizar            3           3.0       1.0
9          setir            3           3.0       1.0

=== Sustantivos más sobrerrepresentados en ciencia ===
         palabra  freq_corpus  freq_ciencia  relacion
0      bioetanol            9           9.0       1.0
1    biopolímero            9           9.0       1.0
2      electrodo            8           8.0       1.0
3      anhídrido            8           8.0       1.0
4         estent            7

In [40]:
# Filtrar filas donde freq_ciencia > 100
tabla_sustantivos[tabla_sustantivos['freq_ciencia'] > 30].head(50)

,palabra,freq_corpus,freq_ciencia,relacion
1637,flora,86,68.0,0.790698
1646,carbono,192,146.0,0.760417
1706,deforestación,447,333.0,0.744966
1718,ecoturismo,63,45.0,0.714286
1727,biodiversidad,363,257.0,0.707989
1728,conservación,345,242.0,0.701449
1735,embalse,45,31.0,0.688889
1739,fauna,181,123.0,0.679558
1740,hábitat,80,54.0,0.675000
1741,ciénaga,58,39.0,0.672414


In [23]:
tabla_adjetivos.head(50)

,palabra,freq_corpus,freq_ciencia,relacion
0,hidrológico,12,12.0,1.0
1,placebo,11,11.0,1.0
2,carbónico,9,9.0,1.0
3,causante,5,5.0,1.0
4,predador,5,5.0,1.0
5,azufre,4,4.0,1.0
6,peluche,4,4.0,1.0
7,nutabe,4,4.0,1.0
8,ensombrecido,4,4.0,1.0
9,humedal,4,4.0,1.0
